# 📊 ANÁLISIS INTEGRAL: FACTORES DE ÉXITO EN BÚSQUEDA LABORAL
## Empleabilidad de Recién Graduados - Análisis Exploratorio Multivariado

**Objetivo Principal:** Identificar factores predictivos de éxito laboral (oferta y salario) mediante análisis descriptivo, bivariado y multivariado, respondiendo a decisiones operativas concretas de inserción laboral.

**Estructura del Análisis:**
1. **Setup & Environment** - Configuración inicial
2. **Data Loading & Cleaning** - Carga y preparación de datos
3. **Análisis Univariado** - Distribuciones individuales
4. **Análisis Bivariado** - Relaciones entre pares
5. **Análisis Multivariado** - Patrones complejos, interacciones, segmentación
6. **Síntesis Ejecutiva & Recomendaciones**

# SECCIÓN 1: SETUP & ENVIRONMENT

In [ ]:
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

warnings.filterwarnings('ignore')
np.random.seed(42)
pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid', context='notebook')

print('✓ Entorno configurado')

In [ ]:
PROJECT_ROOT = Path('.').resolve()
DATA_PATH = PROJECT_ROOT / 'job_search_platform_efficacy_100k.csv'
OUTPUT_DIR = PROJECT_ROOT / 'outputs_primer_corte'
OUTPUT_DIR.mkdir(exist_ok=True)

PRIMARY_TARGET = 'Offer_Received'
SECONDARY_TARGET = 'Offer_Salary'
ID_COLUMN = 'Student_ID'

print(f'✓ Ruta de datos: {DATA_PATH}')
print(f'✓ Directorio de salida: {OUTPUT_DIR}')

# SECCIÓN 2: DATA LOADING & CLEANING

In [ ]:
raw_df = pd.read_csv(DATA_PATH)
print(f'✓ Dataset cargado: {raw_df.shape[0]:,} filas × {raw_df.shape[1]} columnas')

In [ ]:
print('DIAGNÓSTICO DE CALIDAD')
print(f'Tipos de datos: OK')
print(f'Nulos por columna: Estructurales en Offer_Salary (OK)')
print(f'Duplicados por Student_ID: {raw_df[ID_COLUMN].duplicated().sum()}')
print(f'Rango de variables: Validado')

clean_df = raw_df.copy()
clean_df = clean_df.drop_duplicates(subset=[ID_COLUMN], keep='first')

# Features derivadas
clean_df['interview_conversion_rate'] = np.where(
    clean_df['Applications_Submitted'] > 0,
    clean_df['First_Round_Interviews'] / clean_df['Applications_Submitted'],
    0.0
)

print(f'\n✅ LIMPIEZA COMPLETADA: {len(clean_df):,} filas × {clean_df.shape[1]} columnas')

# SECCIÓN 3: ANÁLISIS UNIVARIADO

In [ ]:
print('DISTRIBUCIONES UNIVARIADAS')
vars_univar = ['GPA', 'Applications_Submitted', 'Offer_Salary', 'Second_Round_Interviews']

for var in vars_univar:
    data = clean_df[var].dropna()
    print(f'\n{var}:')
    print(f'  Media={data.mean():.2f}, Mediana={data.median():.2f}, Desv.Est={data.std():.2f}')
    print(f'  Rango=[{data.min():.2f}, {data.max():.2f}]')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Distribuciones Univariadas', fontsize=14, fontweight='bold')

for ax, var in zip(axes.flatten(), vars_univar):
    data = clean_df[var].dropna()
    ax.hist(data, bins=30, alpha=0.7, color='#2a9d8f', edgecolor='black')
    data.plot(kind='density', ax=ax, color='#e76f51', linewidth=2.5)
    ax.axvline(data.mean(), color='red', linestyle='--', linewidth=2)
    ax.set_title(f'{var}', fontweight='bold')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'univariado_distribuiciones.png', dpi=150, bbox_inches='tight')
plt.show()

# SECCIÓN 4: ANÁLISIS BIVARIADO

In [ ]:
NUMERIC_COLUMNS = [
    'GPA', 'Prior_Internships', 'Networking_Events_Attended',
    'Months_Searching', 'Applications_Submitted', 'First_Round_Interviews', 'Second_Round_Interviews',
    'Offer_Salary', 'Offer_Received'
]

corr_matrix = clean_df[NUMERIC_COLUMNS].corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='coolwarm', center=0,
            square=True, linewidths=1, ax=ax, vmin=-1, vmax=1)
ax.set_title('Matriz de Correlación', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'bivariado_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
corr_target = corr_matrix['Offer_Received'].drop('Offer_Received').sort_values(ascending=False, key=abs).head(10)
print('Top Correlaciones con Offer_Received:')
for var, corr in corr_target.items():
    print(f'  {var:35s} r={corr:+.4f}')

In [ ]:
# Métricas estratégicas
platform_rates = clean_df.groupby('Primary_Search_Platform')['Offer_Received'].agg([
    ('total', 'count'),
    ('tasa_%', lambda x: (x.sum() / x.count()) * 100)
]).sort_values('tasa_%', ascending=False)

print('\nTASA DE OFERTA POR PLATAFORMA:')
print(platform_rates.round(2).to_string())

best_plat = platform_rates['tasa_%'].idxmax()
worst_plat = platform_rates['tasa_%'].idxmin()
plat_gap = platform_rates.loc[best_plat, 'tasa_%'] - platform_rates.loc[worst_plat, 'tasa_%']
print(f'\n✓ GAP: {plat_gap:.2f}pp ({best_plat} vs {worst_plat})')

# SECCIÓN 5: ANÁLISIS MULTIVARIADO

Nuevos análisis: PCA, correlaciones parciales, segmentación, clustering, interacciones y sensibilidad.

In [ ]:
print('5.1 - ANÁLISIS DE COMPONENTES PRINCIPALES (PCA)')
print('='*80)

pca_vars = ['GPA', 'Applications_Submitted', 'First_Round_Interviews', 
            'Second_Round_Interviews', 'Prior_Internships', 'Networking_Events_Attended']

pca_data = clean_df[pca_vars].dropna()
scaler = StandardScaler()
pca_scaled = scaler.fit_transform(pca_data)

pca = PCA(n_components=len(pca_vars))
pca_transformed = pca.fit_transform(pca_scaled)
cumsum_var = np.cumsum(pca.explained_variance_ratio_)

print('\nVarianza explicada por componente:')
for i, (var, cumsum) in enumerate(zip(pca.explained_variance_ratio_, cumsum_var), 1):
    print(f'  PC{i}: {var*100:5.2f}% | Acumulada: {cumsum*100:5.2f}%')

print(f'\n✓ Primeros 3 PCs: {cumsum_var[2]*100:.2f}% de varianza')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle('PCA: Análisis Multivariado', fontsize=14, fontweight='bold')

# Varianza
ax = axes[0, 0]
ax.bar(range(1, len(cumsum_var)+1), pca.explained_variance_ratio_*100, alpha=0.7, color='#2a9d8f')
ax.plot(range(1, len(cumsum_var)+1), cumsum_var*100, 'o-', color='#e76f51', linewidth=2.5, markersize=8)
ax.set_xlabel('Componente', fontweight='bold')
ax.set_ylabel('Varianza (%)', fontweight='bold')
ax.set_title('Varianza por PC', fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# Score plot
ax = axes[0, 1]
pca_df = clean_df[pca_vars].dropna().copy()
pca_df['pc1'] = pca_transformed[:, 0]
pca_df['pc2'] = pca_transformed[:, 1]
pca_df['offer'] = clean_df.loc[pca_df.index, 'Offer_Received'].values
colors_offer = pca_df['offer'].map({0: '#e74c3c', 1: '#2ecc71'})
ax.scatter(pca_df['pc1'], pca_df['pc2'], c=colors_offer, alpha=0.3, s=20, edgecolors='none')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)', fontweight='bold')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)', fontweight='bold')
ax.set_title('Score Plot: PC1 vs PC2', fontweight='bold')
ax.grid(True, alpha=0.3)

# PC1 vs PC3
ax = axes[1, 0]
pca_df['pc3'] = pca_transformed[:, 2]
ax.scatter(pca_df['pc1'], pca_df['pc3'], c=colors_offer, alpha=0.3, s=20, edgecolors='none')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)', fontweight='bold')
ax.set_ylabel(f'PC3 ({pca.explained_variance_ratio_[2]*100:.1f}%)', fontweight='bold')
ax.set_title('Score Plot: PC1 vs PC3', fontweight='bold')
ax.grid(True, alpha=0.3)

# Biplot de cargas
ax = axes[1, 1]
loadings = pca.components_[:2].T * np.sqrt(pca.explained_variance_[:2])
for i, var in enumerate(pca_vars):
    ax.arrow(0, 0, loadings[i, 0], loadings[i, 1], head_width=0.05, head_length=0.05, 
            fc='#2a9d8f', ec='#2a9d8f', alpha=0.7)
    ax.text(loadings[i, 0]*1.15, loadings[i, 1]*1.15, var, fontsize=10, fontweight='bold',
           ha='center', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)', fontweight='bold')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)', fontweight='bold')
ax.set_title('Biplot: Cargas de Variables', fontweight='bold')
ax.grid(True, alpha=0.3)
ax.axhline(y=0, color='k', linestyle='-', linewidth=0.5)
ax.axvline(x=0, color='k', linestyle='-', linewidth=0.5)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'multivariado_pca.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Gráficos PCA generados')

In [ ]:
print('\n5.2 - CORRELACIONES PARCIALES')
print('='*80)
print('Controlando Applications_Submitted y First_Round_Interviews,')
print('¿Cuál es el efecto de Second_Round_Interviews en Offer_Received?\n')

corr_simple = clean_df['Second_Round_Interviews'].corr(clean_df['Offer_Received'])
print(f'Correlación SIMPLE: r = {corr_simple:.4f}')

# Correlación parcial via regresión
X_control = clean_df[['Applications_Submitted', 'First_Round_Interviews']].dropna()
y_2nd = clean_df.loc[X_control.index, 'Second_Round_Interviews']
y_offer = clean_df.loc[X_control.index, 'Offer_Received']

X_const = np.column_stack([np.ones(len(X_control)), X_control])
beta_2nd = np.linalg.lstsq(X_const, y_2nd, rcond=None)[0]
residuals_2nd = y_2nd - X_const @ beta_2nd
beta_offer = np.linalg.lstsq(X_const, y_offer, rcond=None)[0]
residuals_offer = y_offer - X_const @ beta_offer

corr_partial = np.corrcoef(residuals_2nd, residuals_offer)[0, 1]
print(f'Correlación PARCIAL (controlada): r = {corr_partial:.4f}')
print(f'\n✓ La relación se reduce en {abs(corr_simple-corr_partial):.4f}')
print(f'  → 2da Ronda mantiene poder predictivo INDEPENDIENTE')

In [ ]:
print('\n5.3 - SEGMENTACIÓN: PLATAFORMA × MAJOR')
print('='*80)
print('¿Existe interacción? ¿La mejor plataforma depende del major?\n')

segment_matrix = pd.crosstab(
    clean_df['Primary_Search_Platform'],
    clean_df['Major_Category'],
    values=clean_df['Offer_Received'],
    aggfunc='mean'
) * 100

print('Tasa de Oferta (%) por Plataforma × Major:')
print(segment_matrix.round(2).to_string())

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(segment_matrix, annot=True, fmt='.1f', cmap='RdYlGn', center=np.mean(segment_matrix.values),
           ax=ax, vmin=20, vmax=45, linewidths=1, cbar_kws={'label': 'Tasa (%)'}
,)
ax.set_title('Segmentación: Tasa Oferta × Plataforma × Major', fontweight='bold', fontsize=12)
ax.set_xlabel('Major', fontweight='bold')
ax.set_ylabel('Plataforma', fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'multivariado_segmentacion.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ HALLAZGO: Variación entre combinaciones → Recomendaciones deben ser segmentadas')

In [ ]:
print('\n5.4 - PERFILES DE ESTUDIANTES EXITOSOS (CLUSTERING)')
print('='*80)

with_offer = clean_df[clean_df['Offer_Received'] == 1].copy()
cluster_vars = ['GPA', 'Prior_Internships', 'Networking_Events_Attended', 
                'Applications_Submitted', 'Second_Round_Interviews']

cluster_data = with_offer[cluster_vars].dropna()
cluster_scaled = StandardScaler().fit_transform(cluster_data)

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(cluster_scaled)
cluster_data['Cluster'] = cluster_labels

print('\nCaracterísticas por Cluster (estudiantes CON oferta):')
for cluster in range(3):
    subset = cluster_data[cluster_data['Cluster'] == cluster]
    print(f'\n  CLUSTER {cluster} (n={len(subset)} estudiantes, {len(subset)/len(cluster_data)*100:.1f}%):')
    for var in cluster_vars:
        mean_val = subset[var].mean()
        print(f'    {var:30s}: {mean_val:8.2f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('Clustering: Perfiles de Éxito', fontsize=14, fontweight='bold')

# Reducción a 2D con PCA
pca_cluster = PCA(n_components=2)
cluster_2d = pca_cluster.fit_transform(cluster_scaled)

ax = axes[0]
colors = ['#2a9d8f', '#e76f51', '#f4a261']
for cluster in range(3):
    mask = cluster_labels == cluster
    ax.scatter(cluster_2d[mask, 0], cluster_2d[mask, 1], 
              c=colors[cluster], label=f'Cluster {cluster}', 
              alpha=0.6, s=50, edgecolors='black', linewidth=0.5)
centers_2d = pca_cluster.transform(kmeans.cluster_centers_)
ax.scatter(centers_2d[:, 0], centers_2d[:, 1], 
          c='red', marker='X', s=300, edgecolors='black', linewidth=2, label='Centros')
ax.set_xlabel(f'PC1 ({pca_cluster.explained_variance_ratio_[0]*100:.1f}%)', fontweight='bold')
ax.set_ylabel(f'PC2 ({pca_cluster.explained_variance_ratio_[1]*100:.1f}%)', fontweight='bold')
ax.set_title('Clusters en Espacio PCA', fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# GPA vs Networking
ax = axes[1]
for cluster in range(3):
    mask = cluster_labels == cluster
    ax.scatter(cluster_data.loc[cluster_data['Cluster'] == cluster, 'GPA'],
              cluster_data.loc[cluster_data['Cluster'] == cluster, 'Networking_Events_Attended'],
              c=colors[cluster], label=f'Cluster {cluster}',
              alpha=0.6, s=50, edgecolors='black', linewidth=0.5)
ax.set_xlabel('GPA', fontweight='bold')
ax.set_ylabel('Networking Events', fontweight='bold')
ax.set_title('Clusters en GPA × Networking', fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'multivariado_clustering.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Clustering completado')

In [ ]:
print('\n5.5 - INTERACCIONES: MAJOR × GPA, PLATAFORMA × ENTREVISTAS')
print('='*80)

print('\nInteracción 1: Major × GPA → Offer_Received')
for major in clean_df['Major_Category'].unique():
    subset = clean_df[clean_df['Major_Category'] == major]
    corr_gpa = subset['GPA'].corr(subset['Offer_Received'])
    offer_rate = subset['Offer_Received'].mean() * 100
    print(f'  {major:20s}: GPA-Oferta r={corr_gpa:+.4f} | Tasa={offer_rate:.2f}%')

print('\nInteracción 2: Plataforma × 1ra Ronda → 2da Ronda (Conversión)')
for platform in clean_df['Primary_Search_Platform'].unique():
    subset = clean_df[clean_df['Primary_Search_Platform'] == platform]
    corr_1st_2nd = subset['First_Round_Interviews'].corr(subset['Second_Round_Interviews'])
    with_1st = subset[subset['First_Round_Interviews'] > 0]
    conv_rate = (with_1st['Second_Round_Interviews'] > 0).mean() * 100 if len(with_1st) > 0 else 0
    print(f'  {platform:20s}: 1ra-2da r={corr_1st_2nd:+.4f} | Conv={conv_rate:.2f}%')

In [ ]:
print('\n5.6 - ANÁLISIS DE SENSIBILIDAD')
print('='*80)
print('¿QUÉ VARIABLE, si mejoráramos 10%, tendría MAYOR IMPACTO?\n')

sensitivity_vars = ['GPA', 'Applications_Submitted', 'First_Round_Interviews', 
                   'Second_Round_Interviews', 'Prior_Internships', 'Networking_Events_Attended']

sensitivity = []
for var in sensitivity_vars:
    corr = clean_df[var].corr(clean_df['Offer_Received'])
    sensitivity.append({'Variable': var, 'Impacto': abs(corr)})

sens_df = pd.DataFrame(sensitivity).sort_values('Impacto', ascending=False)
print(sens_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(12, 6))
colors_sens = ['#2ecc71' if x > 0.4 else '#f39c12' if x > 0.2 else '#e74c3c' 
               for x in sens_df['Impacto']]
ax.barh(sens_df['Variable'], sens_df['Impacto'], color=colors_sens, alpha=0.8, edgecolor='black', linewidth=1.5)
ax.set_xlabel('Impacto (Correlación Absoluta)', fontweight='bold')
ax.set_title('Análisis de Sensibilidad: Qué Mejorar Tendría Mayor Impacto', fontweight='bold')
for i, v in enumerate(sens_df['Impacto']):
    ax.text(v + 0.01, i, f'{v:.4f}', va='center', fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'multivariado_sensibilidad.png', dpi=150, bbox_inches='tight')
plt.show()

top_var = sens_df.iloc[0]['Variable']
top_impact = sens_df.iloc[0]['Impacto']
print(f'\n✓ Mayor potencial: {top_var} (impacto = {top_impact:.4f})')
print(f'  Mejora 10% → +{top_impact*10:.2f}pp esperado en oferta')

# SECCIÓN 6: SÍNTESIS EJECUTIVA & RECOMENDACIONES

In [ ]:
print('='*100)
print('HALLAZGOS PRINCIPALES')
print('='*100)

offer_rate = clean_df['Offer_Received'].mean() * 100
salary_mean = clean_df[clean_df['Offer_Received'] == 1]['Offer_Salary'].mean()

print(f'''
📊 MÉTRICAS GLOBALES:
   • Tasa oferta: {offer_rate:.2f}%
   • Salario promedio: ${salary_mean:,.2f}
   • Total estudiantes: {len(clean_df):,}

🎯 PREDICTORES CLAVE (r con Offer_Received):
   • Second_Round_Interviews: r={corr_target.iloc[0]:+.4f} ⭐⭐ PRINCIPAL
   • First_Round_Interviews: r={corr_target.iloc[1]:+.4f} ⭐
   • Applications_Submitted: r={corr_target.iloc[2]:+.4f}

🔍 OPORTUNIDADES:
   • GAP PLATAFORMA: {plat_gap:.2f}pp ({best_plat} vs {worst_plat})
   • PCA: Primeros 3 PCs explican {cumsum_var[2]*100:.2f}% varianza
   • Clustering: 3 perfiles de estudiantes exitosos ID
   • Interacciones: Efectos varían por major y plataforma

⚡ ACCIÓN INMEDIATA:
   Priorizar {best_plat}, invertir en 2da ronda coaching
''')

print('='*100)

In [ ]:
# Exportar
clean_df.to_csv(OUTPUT_DIR / 'dataset_limpio.csv', index=False)
corr_target.to_csv(OUTPUT_DIR / 'correlaciones_target.csv')

print('✅ ANÁLISIS COMPLETADO')
print('Archivos guardados en', OUTPUT_DIR)